# 02 · Fock 表示新手教程

**Fock 表示**把态写成光子数基底 $\{|n\rangle\}$ 上的振幅（或密度矩阵 $\rho$）。

本教程只动 **`cvsim.fock`**：截断 → 门 → PNRD → loss→ρ → Wigner → Homodyne。

配套笔记：`01-Fock表示原理.md`、`04-…` 四问篇。

## 1. 这是啥 / 为啥用

- 你想问：「测到 0、1、2… 光子的概率是多少？」→ Fock 最直接。
- 非高斯门（Kerr）、截断下的精确幺正，也走 Fock。
- **代价：** 截断 $N$，$m$ 模维度 $\sim N^m$。本包教学用 **1–2 模**。

**一句话：** 要光子数 / 非高斯，用 Fock；模一多就痛。

## 2. 约定

与 Gaussian 同一物理：$\hbar=1$，位移 $\sqrt{2}$ 约定。

额外：

- `FockState`：纯态振幅；2 模时 `amps` 形状 $(N,N)$
- `FockDensity`：混态 $\rho$（loss 之后）
- **截断不是物理墙**，是数值近似——$N$ 太小会错

In [ ]:
# 从仓库根启动 Jupyter 最稳；若在 tutorials/ 里打开，这里兜底加路径
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "cvsim").is_dir():
    ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei']  # 中文支持
matplotlib.rcParams['axes.unicode_minus'] = False    # 负号显示
print("repo root:", ROOT)
print("numpy", np.__version__)

In [ ]:
from cvsim.fock import (
    FockState,
    beamsplitter,
    displace,
    homodyne_condition,
    homodyne_mean,
    loss,
    mean_photon,
    norm,
    pnrd_probs,
    squeeze,
    trace,
)
from cvsim.wigner import wigner_fock

## 3. 最小闭环：截断挤压

解析：$\langle n\rangle = \sinh^2 r$。Fock 里用截断幺正近似——**N 越大越准**。

In [ ]:
r = 0.5
n_exact = float(np.sinh(r) ** 2)
print(f"target <n> = sinh^2({r}) = {n_exact:.6f}")
for N in [4, 6, 8, 12, 20]:
    st = squeeze(FockState.vacuum(N), r=r)
    err = abs(mean_photon(st) - n_exact)
    print(f"  N={N:3d}  <n>={mean_photon(st):.6f}  |err|={err:.3e}  ||ψ||={norm(st):.6f}")

## 4. 数字检查：PNRD 与双模 BS

$|10\rangle$ 过 50/50 BS → 两端单光子概率各约 $1/2$。

In [ ]:
# 单模：|1> 的光子数分布
st1 = FockState.fock(1, cutoff=8)
print("|1> pnrd:", pnrd_probs(st1))

# 双模 |10> → BS（amps 形状 (N,N)）
psi = FockState.fock2(1, 0, cutoff=6)
psi = beamsplitter(psi, theta=np.pi / 4)
p10 = abs(psi.amps[1, 0]) ** 2
p01 = abs(psi.amps[0, 1]) ** 2
print("|c10|^2, |c01|^2 ≈", float(p10), float(p01), "  (expect ~0.5 each)")

## 5a. 损耗 → 密度矩阵

纯态 $|1\rangle$ 经透射率 $T$ 的纯损耗：$\rho_{00}\approx 1-T$，$\rho_{11}\approx T$。

**混态**用 `FockDensity`；`trace(ρ)≈1`。

In [ ]:
T = 0.3
rho = loss(FockState.fock(1, cutoff=10), T=T)
print("type:", type(rho).__name__)
print("Tr ρ =", trace(rho))
print("ρ[0,0], ρ[1,1] ≈", float(rho.rho[0, 0].real), float(rho.rho[1, 1].real))
print("expect ~", 1 - T, T)

## 5b. Wigner 与 Homodyne

真空：$W(0,0)=1/\pi$。$|1\rangle$ 中心可负（非经典）。

**诚实：** Fock 的 `homodyne_condition` 是 **截断空间里 $x_\varphi$ 本征投影**，  
**不是** Gaussian 那套 Kalman 后验。先验振幅几乎被扔掉，后验 ≈ 最近本征矢。

In [ ]:
N = 20
vac = FockState.vacuum(N)
one = FockState.fock(1, N)
w0 = wigner_fock(vac, 0.0, 0.0)
w1 = wigner_fock(one, 0.0, 0.0)
print("W_vac(0,0) =", w0, "  expect", 1 / np.pi)
print("W_|1|(0,0) =", w1, "  (should be negative)")

print("homodyne mean |1> φ=0:", homodyne_mean(one, phi=0.0))  # ~0 by parity
# 条件：投到最近 x 本征态（教学）
post = homodyne_condition(one, mode=0, phi=0.0, outcome=0.0)
print("after condition: type", type(post).__name__, "norm", norm(post))

## 6. 诚实边界 + 何时换表示

**Fock 适合**

- PNRD、小 cutoff 精确门、loss→ρ、单模 Wigner

**Fock 不适合 / 本包限制**

- $m\ge 3$、大 cutoff
- 2 模 ρ 上门 / Wigner / Homodyne（多数未做）
- 大规模高斯电路 → **Gaussian**
- 大振幅 cat/GKP 全貌 → **Bosonic** 高斯叠加更省

下一本：`03_bosonic_beginner.ipynb`。

## 自检

In [ ]:
r = 0.5
st = squeeze(FockState.vacuum(20), r=r)
assert abs(mean_photon(st) - np.sinh(r) ** 2) < 1e-3
rho = loss(FockState.fock(1, 12), T=0.4)
assert abs(trace(rho) - 1.0) < 1e-10
assert abs(rho.rho[0, 0].real - 0.6) < 1e-8
assert abs(rho.rho[1, 1].real - 0.4) < 1e-8
assert abs(wigner_fock(FockState.vacuum(30), 0.0, 0.0) - 1 / np.pi) < 1e-6
assert wigner_fock(FockState.fock(1, 30), 0.0, 0.0) < 0
print("T2 self-check OK")